# Epics Variables with Readbacks

In [17]:
from bluesky import RunEngine
from bluesky.callbacks import LiveTable
from bluesky.plans import count, list_scan
from bluesky import plan_stubs as bps
from bluesky import plans as bp

RE = RunEngine()

In [20]:
import numpy as np
from ophyd_async.core import init_devices
from ophyd_async.epics.core import epics_signal_r
from ophyd.signal import EpicsSignal, EpicsSignalRO

# PVAccess (hard - higher performance - ophyd async)
with init_devices():
    etrace = epics_signal_r(
        np.ndarray,
        "pva://ELECTRON-DAQ:trace",
        name="etrace")

    ptrace = epics_signal_r(
        np.ndarray,
        "pva://PROTON-DAQ:trace",
        name="ptrace")

# Channel Access (easy - lower performance - ophyd sync)
trig_count = EpicsSignalRO("ELECTRON:trigger:count", name="trig_count")

powers = EpicsSignal(
    read_pv="LASER:powers",
    write_pv="LASER:powers:set",
    name="powers",
)

In [21]:
powers.get()

array([255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255], dtype=int32)

In [25]:
def condition_pulse(pulse):
    """ Checks for mistakes in the pulse array and casts as uint8
    Expects an input of 100 elements already ranging from 0 to 255 """
    
    # Data validity checks
    assert(isinstance(pulse, np.ndarray))
    assert(np.ndim(pulse) == 1)
    assert(len(pulse) == 100)
    assert(np.max(pulse) <= 255)
    assert(np.min(pulse) >= 0)
    if (np.count_nonzero(pulse - pulse.round() > 0)):
        raise Exception("Pulse array contains fractional values (should be only integers, from 0 to 255).")

    # Data type conversion to uint8
    pulse_uint8 = np.uint8(pulse)
    
    return pulse_uint8

In [28]:
pulse_flat = np.ones(100)*255
pulse_flat = condition_pulse(pulse_flat)

powers.set(pulse_flat)

Status(obj=EpicsSignal(read_pv='LASER:powers', name='powers', value=array([255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255], dtype=int32), timestamp=1781827741.119337, auto_monitor=False, string=False, write_pv='LASER:powers:set', limits=False, put_complete=False), done=False, success=False)

In [29]:
powers.get()

array([255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255], dtype=int32)

In [30]:
RE(bps.abs_set(powers, pulse_flat, wait=True))

()